# Ingest JDBC

**Author:** Databricks Ingestion Team  
**Version:** 1.0  
**Last Updated:** 2026-03-31

This notebook ingests data from JDBC sources. **Default: Incremental Load** (only new data is ingested). Set the load type widget to 'full' for a full reload.

---

**Parameters:**
- `jdbc_url`: JDBC connection string
- `table_name`: Source table name
- `output_table`: Destination table name
- `load_type`: 'incremental' (default) or 'full'

---

**Instructions:**
1. Set the JDBC URL, source table, output table, and load type using the widgets below.
2. Run all cells to ingest data as per the selected load type.


In [ ]:
# Databricks widgets for parameterization
import pyspark.sql.functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ingest_jdbc")

dbutils.widgets.text("jdbc_url", "", "JDBC URL")
dbutils.widgets.text("table_name", "", "Source Table Name")
dbutils.widgets.text("output_table", "bronze_table", "Output Table")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"], "Load Type")
jdbc_url = dbutils.widgets.get("jdbc_url")
table_name = dbutils.widgets.get("table_name")
output_table = dbutils.widgets.get("output_table")
load_type = dbutils.widgets.get("load_type")

try:
    df = (spark.read.format("jdbc")
          .option("url", jdbc_url)
          .option("dbtable", table_name)
          .load())
    logger.info(f"Loaded {df.count()} records from {table_name}")
    if load_type == "full":
        df.write.format("delta").mode("overwrite").saveAsTable(output_table)
        logger.info(f"Full load: Overwrote {output_table}")
    else:
        df.write.format("delta").mode("append").saveAsTable(output_table)
        logger.info(f"Incremental load: Appended to {output_table}")
    print(f"Ingestion ({load_type}) successful.")
except Exception as e:
    logger.error(f"Error in ingestion: {e}")
    print(f"Error: {e}")


## JDBC Option Builder
Function to validate and build JDBC options for Spark.

In [ ]:
def build_jdbc_options(jdbc_profile: dict, source: dict) -> dict:
    # ...existing code...

## Parallel Read Options
Function to merge parallel read options for large tables.

In [ ]:
def _apply_parallel_read_options(options: dict, extra_options: dict) -> dict:
    # ...existing code...

## JDBC Batch Ingestion
Function to read a JDBC table or query into a Spark DataFrame.

In [ ]:
def ingest_jdbc_batch(spark, jdbc_profile: dict, source: dict, extra_options: dict | None = None):
    # ...existing code...

---

## Validation & Testing

Below, we validate the notebook logic with a simple assertion to ensure data was ingested. Add more tests as needed for your data.


In [ ]:
# Simple validation
try:
    df_test = spark.table(output_table)
    assert df_test.count() > 0, "No records found in output table."
    print("Validation passed: Data ingested.")
except Exception as e:
    print(f"Validation failed: {e}")


---

## Resource Cleanup

Unpersist DataFrames and clean up resources if needed to avoid memory issues in production workflows.

In [ ]:
# Resource cleanup
if 'df_test' in locals():
    df_test.unpersist(blocking=True)
    print("Resources cleaned up.")
